# Step 1 — Safely append an `exp_day` to an existing project

**Purpose:** add only new recording stems and Ilastik reference BMPs so the additional recordings can later be registered to their historical reference and used in the manual lesion assessment GUI.

This notebook **does not** create a new project, edit the main YAML, or recompute existing recordings. `Step1_ProjectCreation.ipynb` is intended for initial project creation and must not be rerun merely to add a day.

Before starting, place the current `common.py`, `step1_project_creation.py`, and `step1_add_exp_day.py` in the same `src/` directory. Verify the existing config and the exact day-folder label.


## 1. Import the current modules


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
for candidate in (NOTEBOOK_DIR / "src", NOTEBOOK_DIR.parent / "src", NOTEBOOK_DIR):
    if all((candidate / name).is_file() for name in (
        "common.py", "step1_project_creation.py", "step1_add_exp_day.py"
    )):
        SRC_DIR = candidate
        break
else:
    raise FileNotFoundError("Поместите три актуальных .py файла в src/ или рядом с ноутбуком")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from step1_add_exp_day import plan_add_exp_day, commit_add_exp_day
print("Source folder:", SRC_DIR)


## 2. Specify the existing configuration and new `exp_day`

`NEW_EXP_DAY` must exactly match the folder name under `discovery.raw_root` (for example, `7d-PT`). `CONFIG_PATH` must point to the **existing project's main YAML** containing `project.directory_to_save_main_analysis`.


In [ ]:
CONFIG_PATH = Path(r"config.yaml")  # Verify this path.
NEW_EXP_DAY = "PT"  # Replace with the exact folder name, e.g., "7d-PT".

if NEW_EXP_DAY == "NEW_EXP_DAY":
    raise ValueError("Укажите фактическое имя нового exp_day")
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Конфиг не найден: {CONFIG_PATH}")
print("Config:", CONFIG_PATH)
print("New exp_day:", NEW_EXP_DAY)


## 3. Preflight — no new reference BMPs or manifest update

Check the existing manifest, `stem_keys` compatibility, `raw.rotation`/`raw.scale` geometry, historical mouse grouping and reference selection, availability of new frames, and absence of stem or BMP-path collisions. A separate audit report is written under `step1_project/incremental_step1_logs/`.

**Note:** preflight does write its audit files; it does not modify the historical manifest or reference BMPs.


In [ ]:
plan = plan_add_exp_day(CONFIG_PATH, NEW_EXP_DAY)
print(json.dumps(plan.summary(), indent=2, ensure_ascii=False))
if not plan.new_rows.empty:
    show_cols = [c for c in ["stem", "mouse_id", "exp_day", "state", "protocol", "img_dir"] if c in plan.new_rows]
    display(plan.new_rows[show_cols])
print("Report:", plan.log_dir / "manifest_preflight.csv")


## 4. Inspect the new-recording list

Check mouse identifiers, `stem`, `state`, `protocol`, and the number of new BMPs. If the selection is incorrect, **do not commit**; correct the raw-data organization or configuration first.


In [ ]:
if (plan.log_dir / "manifest_preflight.csv").is_file():
    report = pd.read_csv(plan.log_dir / "manifest_preflight.csv")
    display(report.groupby("status", dropna=False).size().rename("n_recordings").reset_index())
    display(report[report["status"].ne("included")].head(30))
else:
    print("No preflight report found")


In [ ]:
report

## 5. Commit — only after reviewing the preflight

**Warning: the current code cell has `COMMIT = True`.** Running it will attempt to append new recordings. Set `COMMIT = False` yourself for a non-committing inspection; switch back to `True` only after confirming the plan.

On a successful commit, the original manifest is backed up; reference BMPs are produced only for new stems; the historical CSV bytes remain an exact prefix of the appended manifest. If the manifest changes between preflight and commit, the operation fails closed.


In [ ]:
COMMIT = True  # Set True only after successfully reviewing the preflight above.

if not COMMIT:
    print("PREVIEW ONLY. Manifest и BMP старого проекта не изменены.")
else:
    result = commit_add_exp_day(plan)
    print(json.dumps(result, indent=2, ensure_ascii=False))


## 6. Verify the commit

New stems should appear in `manifest_recordings.csv`. A subsequent preflight for the same `exp_day` should report `new_stems: []`.


In [ ]:
if COMMIT:
    updated = pd.read_csv(plan.existing_manifest)
    print("Manifest rows:", len(updated))
    added = updated[updated["stem"].astype(str).isin(plan.new_rows["stem"].astype(str))]
    cols = [c for c in ["stem", "ref_505_path", "ref_656_path", "prob_505_exists", "prob_656_exists"] if c in added]
    display(added[cols])
    repeat = plan_add_exp_day(CONFIG_PATH, NEW_EXP_DAY)
    print("Remaining new stems after commit:", len(repeat.new_rows))
else:
    print("Commit не выполнен; проверка пропущена")


## After Step 1

1. Export Ilastik probability `.h5` files for the new reference BMPs using the same wavelengths and settings as historical recordings (e.g., 505 and 656 nm in the current configuration).
2. **Do not rerun the original full Step2 for an existing mouse group.** The standard workflow may rewrite historical results. Use the incremental Step2 procedure, retaining the reference selected in the prior registration.
3. Do not run lesion assessment or Step4 for the new recordings until registration and QC are complete.
